# Wan 2.1 GPU Telegram Worker

Run on a Colab/Kaggle GPU runtime (T4 or better). The open Wan 2.1 1.3B Diffusers weights do not need a paid API key. Free GPU allocation, runtime, memory, Telegram limits and daily throughput are not guaranteed.

**Telegram constraint:** `getUpdates` long polling cannot operate while the same bot has a webhook configured. This worker refuses to start if a webhook is active. Use a dedicated bot for simultaneous Cloudflare and notebook operation, or consciously disable the webhook outside this notebook. Enter the bot token at runtime; never save it in the notebook.

The worker restricts requests to one configured chat ID, processes prompts sequentially, assigns a fresh production ID and deterministic per-job Wan seeds, and lets the repository's creative director choose which scenes receive generated motion. It then uses the shared social-caption, cover-frame, render-QA and Telegram-delivery steps.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/piyushmal13/cinema-video-studio1.git"
WORK_DIR = Path.cwd()
REPO_DIR = WORK_DIR if (WORK_DIR / "scripts" / "generate_assets.py").is_file() else WORK_DIR / "cinema-video-studio1"
if not (REPO_DIR / "scripts" / "generate_assets.py").is_file():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

subprocess.run(["apt-get", "-qq", "update"], check=True)
subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg", "nodejs", "npm"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_DIR / "requirements.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-U", "diffusers>=0.33,<1", "transformers>=4.49,<5", "accelerate>=1.2,<2", "ftfy", "imageio"], check=True)
subprocess.run(["npm", "ci"], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print("Repository and render dependencies are ready at", REPO_DIR)

In [ ]:
import hashlibimport torchfrom diffusers import WanPipelinefrom diffusers.utils import export_to_videoif not torch.cuda.is_available():    raise RuntimeError("Enable a GPU accelerator in the Colab/Kaggle runtime settings, then rerun this cell.")print("GPU:", torch.cuda.get_device_name(0))print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))WAN_MODEL_ID = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"wan_pipe = WanPipeline.from_pretrained(WAN_MODEL_ID, torch_dtype=torch.float16, use_safetensors=True)wan_pipe.enable_model_cpu_offload()wan_pipe.enable_vae_tiling()wan_pipe.enable_vae_slicing()print("Wan 2.1 1.3B loaded. First generation can take longer while weights warm up.")def generate_local_wan_clip(prompt, idx, orientation, out_mp4):    width, height = (832, 480) if orientation == "landscape" else (480, 832)    try:        result = wan_pipe(            prompt=prompt,            negative_prompt="blurry, distorted motion, captions, text, watermark",            height=height,            width=width,            num_frames=33,            num_inference_steps=12,            guidance_scale=5.0,            generator=torch.Generator(device="cuda").manual_seed(                int(hashlib.sha256(                    f"{os.environ.get('CINEMA_JOB_ID', 'wan-default')}\\0{idx}\\0{prompt}".encode()                ).hexdigest()[:8], 16)            ),        )        export_to_video(result.frames[0], out_mp4, fps=16)        valid = Path(out_mp4).is_file() and Path(out_mp4).stat().st_size > 50_000        if not valid:            Path(out_mp4).unlink(missing_ok=True)        return valid    except Exception as error:        Path(out_mp4).unlink(missing_ok=True)        print("Local Wan clip failed; the regular still-image fallback will be used:", error)        return Falseprint("Local Wan clip callback ready.")

In [ ]:
import asyncioimport getpassimport importlibimport jsonimport osimport shutilimport uuidimport requestsfrom scripts import generate_assetsfrom scripts.post_production import run as prepare_social_packagefrom scripts.deliver_video import deliver_videoBOT_TOKEN = getpass.getpass("Telegram bot token (input is hidden): ").strip()ALLOWED_CHAT_ID = input("Allowed Telegram chat ID (required): ").strip()if not BOT_TOKEN or not ALLOWED_CHAT_ID:    raise ValueError("Both a bot token and an allowed chat ID are required.")API = f"https://api.telegram.org/bot{BOT_TOKEN}"webhook_response = requests.get(f"{API}/getWebhookInfo", timeout=15)webhook_response.raise_for_status()if webhook_response.json().get("result", {}).get("url"):    raise RuntimeError("This bot has an active webhook. Use a dedicated polling bot or disable its webhook before starting this worker.")def send_status(chat_id, text):    response = requests.post(        f"{API}/sendMessage",        data={"chat_id": chat_id, "text": text},        timeout=20,    )    response.raise_for_status()async def render_and_deliver(prompt, chat_id):    if len(prompt) > 3500:        send_status(chat_id, "Please keep video prompts to 3,500 characters or fewer.")        return    if prompt.strip().casefold() == "/stop":        return "stop"    job_id = uuid.uuid4().hex    send_status(chat_id, f"Creative director + Wan GPU render started. Job: {job_id[:8]}")    try:        os.environ["PROMPT"] = prompt        os.environ["FORMAT"] = "vertical"        os.environ["CINEMA_JOB_ID"] = job_id        os.environ["CINEMA_MAX_VIDEO_CLIPS"] = "3"        importlib.reload(generate_assets)        generate_assets.RAW_PROMPT = prompt        # The main pipeline now selects opening/middle/ending scenes dynamically.        # Wan replaces only the video-clip provider; all other routing stays shared.        generate_assets.try_generate_ai_video_clip = generate_local_wan_clip        await asyncio.to_thread(generate_assets.main)        video_data_path = REPO_DIR / "public" / "video_data.json"        video_data = json.loads(video_data_path.read_text(encoding="utf-8"))        build_script = "build:landscape" if video_data.get("format") == "landscape" else "build"        await asyncio.to_thread(            subprocess.run,            ["npm", "run", build_script],            cwd=REPO_DIR,            check=True,        )        if build_script == "build:landscape":            shutil.move(REPO_DIR / "out" / "final_16x9.mp4", REPO_DIR / "out" / "final.mp4")        video_path = REPO_DIR / "out" / "final.mp4"        if not video_path.is_file() or not video_path.stat().st_size:            raise FileNotFoundError("Remotion did not create out/final.mp4")        package = await asyncio.to_thread(            prepare_social_package,            video_path,            video_data_path,            REPO_DIR / "out",        )        await asyncio.to_thread(            deliver_video,            video_path,            BOT_TOKEN,            chat_id,            caption=package["caption"],            thumbnail_path=package.get("cover"),        )        send_status(chat_id, f"Delivered successfully. Job: {job_id[:8]}")    except Exception as error:        send_status(chat_id, f"Render failed for job {job_id[:8]}: {str(error)[:700]}")    return Noneasync def poll_telegram():    offset = None    print("Polling started. Send a prompt to the bot; send /stop to stop the worker.")    while True:        params = {"timeout": 25, "allowed_updates": '["message"]'}        if offset is not None:            params["offset"] = offset        try:            response = await asyncio.to_thread(                requests.get,                f"{API}/getUpdates",                params=params,                timeout=35,            )            response.raise_for_status()            updates = response.json().get("result", [])            for update in updates:                offset = update["update_id"] + 1                message = update.get("message", {})                chat_id = str(message.get("chat", {}).get("id", ""))                if chat_id != ALLOWED_CHAT_ID or message.get("from", {}).get("is_bot"):                    continue                prompt = (message.get("text") or message.get("caption") or "").strip()                if not prompt:                    send_status(chat_id, "Send a text prompt or a caption with your video idea.")                    continue                result = await render_and_deliver(prompt, chat_id)                if result == "stop":                    print("Polling stopped by the allowed chat.")                    return        except requests.RequestException as error:            print("Telegram polling request failed; retrying in 5 seconds:", error)            await asyncio.sleep(5)await poll_telegram()